# Churn from point-in-time features

A retailer wants to know which customers will stop buying. At each weekly cutoff, we
predict which customers will make *no* purchase in the next seven days.

Along the way, this notebook shows how to:

- declare point-in-time features in a contract, such as counts and sums over the last few
  weeks, days since the last purchase, the age of each recent purchase, and the weekday;
- choose an encoder for each field: `ZScore` and `Buckets` for numbers, `Vocabulary` for
  categories;
- compute a label from a *future* window, which training reads and inference never does;
- read anchors from a table that also carries the train / val / test split;
- check for leakage, using the generator's knowledge of the best AUROC any model reading
  only past rows can reach (a model that beats it is reading the future);
- score a test set from a saved artifact, and serve one request from rows you pass in.

The model lives in [`churn_model.py`](churn_model.py) next to this notebook. The
contract, the model's input schema and one serving request with its rows are written
next to it too, so you can read them without running anything.

The data comes from the retail generator in `kumo.pytorch.datasets`, which makes synthetic
data modeled on the public H&M fashion-retail dataset. We use its smallest size: 200
customers and about 8,000 purchases. Everything runs on the DiskGraph
engine over local Parquet files; see [engines and data](../../docs/04-engines-and-data.md)
for setting it up.

In [ ]:
import tempfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq
import torch
from churn_model import ChurnModel, churned, rounded, write_json
from torch import nn

from kumo.pytorch import api, datasets

torch.manual_seed(0)
workdir = Path(tempfile.mkdtemp())
data = datasets.hm(workdir / "hm", scale="xs", seed=0)
print(data.metadata.sizes)

## The schema

There are three tables: static `customers` and `articles`, and timestamped
`transactions`. Every table names its time column or declares `NON_TEMPORAL`. Timestamp
column types are left as `None`, so they're read from the Parquet metadata.

In [ ]:
INT64, FLOAT64, STRING = api.DType("int64"), api.DType("float64"), api.DType("string")
DAY = 86_400

schema = api.DataSchema(
    tables=(
        api.TableSpec(
            "customers",
            "customer_id",
            {
                "customer_id": INT64,
                "age": INT64,
                "club_member_status": STRING,
                "fashion_news_frequency": STRING,
            },
            time_column=api.NON_TEMPORAL,
        ),
        api.TableSpec(
            "articles",
            "article_id",
            {"article_id": INT64, "product_type_name": STRING, "listed_at": None},
            time_column="listed_at",
        ),
        api.TableSpec(
            "transactions",
            "transaction_id",
            {
                "transaction_id": INT64,
                "t_dat": None,
                "customer_id": INT64,
                "article_id": INT64,
                "price": FLOAT64,
                "sales_channel_id": INT64,
            },
            time_column="t_dat",
        ),
    ),
    relations=(
        api.RelationSpec(
            "customer_transactions", "customers", "customer_id", "transactions", "customer_id"
        ),
        api.RelationSpec(
            "article_transactions", "articles", "article_id", "transactions", "article_id"
        ),
    ),
)
dataset = api.Dataset(
    {name: api.Parquet(str(data.table(name))) for name in ("customers", "articles", "transactions")}
)

## What one example may know

A window is relative to the anchor's cutoff, in seconds: start excluded, stop included.
`purchases(-30 * DAY, 0)` is the last 30 days up to and including the cutoff. The label
reads `purchases(0, 7 * DAY)`: strictly after the cutoff, so nothing an input sees is
also counted in the label.

- `recency` is the time since the last purchase.
- Distinct purchase times count visits (a basket shares one timestamp).
- Each recent purchase carries its age in days and the weekday it happened on; neither
  appears unless declared.

In [ ]:
def purchases(start: int | None, stop: int) -> api.Rows:
    return api.Rows(("customer_transactions",), api.TimeWindow("t_dat", start, stop))


WINDOWS = {"visits_7d": 7, "visits_30d": 30, "visits_90d": 90}

sample = api.SampleSpec(
    entity=api.EntitySpec("customers", entity_type="customer"),
    entity_values={
        "age": api.Field(column="age"),
        "club": api.Field(column="club_member_status"),
        "news": api.Field(column="fashion_news_frequency"),
        "recency": api.Aggregate(purchases(None, 0), "recency", "t_dat"),
        "spend_30d": api.Aggregate(purchases(-30 * DAY, 0), "sum", "price"),
        **{
            name: api.Aggregate(purchases(-days * DAY, 0), "count_distinct", "t_dat")
            for name, days in WINDOWS.items()
        },
    },
    events={
        "recent": api.EventSpec(
            purchases(None, 0),
            fields={
                "price": api.Field(column="price"),
                "channel": api.Field(column="sales_channel_id"),
                "time": api.Field(column="t_dat"),
                "weekday": api.Derived(
                    operation=api.CalendarPart(part="weekday"), inputs=("time",), dtype=INT64
                ),
                "age_days": api.EventTimeDelta(unit="days", dtype=FLOAT64),
            },
            limit=16,
            order="descending",
        )
    },
    targets={"purchases_next_week": api.Aggregate(purchases(0, 7 * DAY), "count")},
)

## What the model receives

Encoders are chosen per field, not inferred from types. Age is bucketed, counts and
amounts are standardized, categories become IDs. The history keeps one row per purchase,
padded to the longest history in the batch with a validity mask.

In [ ]:
NUMBERS = ("recency", "spend_30d", *WINDOWS)
CATEGORIES = ("club", "news")

batch = api.BatchSpec(
    inputs={
        "customer": api.Input(
            api.EntitySource(),
            api.Fields(
                (
                    api.Encode("age", api.Buckets(bins=8)),
                    *(api.Encode(name, api.ZScore()) for name in NUMBERS),
                    *(api.Encode(name, api.Vocabulary()) for name in CATEGORIES),
                )
            ),
            api.Separate(),
        ),
        "history": api.Input(
            api.EventSource(("recent",)),
            api.Fields(
                (
                    api.Encode("price", api.ZScore()),
                    api.Encode("channel", api.Vocabulary()),
                    api.Encode("weekday", api.Vocabulary()),
                    api.Encode("age_days", api.ZScore()),
                )
            ),
            api.Padded(),
        ),
    },
    targets={"purchases": api.LabelTarget("purchases_next_week", api.Raw())},
    batching=api.Batching(max_samples=128),
)
contract = api.DataLoaderSpec(schema, sample, batch)

anchors = api.AnchorTable(
    source=api.Parquet(str(data.anchors)),
    entity_id="customer_id",
    as_of_time="cutoff",
    sample_id="sample_id",
    split="split",
)
engine = api.DiskGraphEngine(index=workdir / "index")
print(api.check(contract=contract, dataset=dataset, engine=engine, anchors=anchors))

## A model, and training on the train split

`materialize(..., fit_split="train")` fits every encoder on training anchors only; the
validation and test batches reuse that state. The model, `ChurnModel` in
[`churn_model.py`](churn_model.py), is ordinary PyTorch sized from `task.output_schema`.

In [ ]:
with api.prepare(contract, dataset, engine) as plan:
    with plan.materialize(anchors, fit_split="train") as task:
        model = ChurnModel(task.output_schema, NUMBERS, CATEGORIES)
        optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
        for epoch in range(8):
            model.train()
            losses = []
            for b in task.batches(split="train", epoch=epoch, shuffle=True, seed=0):
                loss = nn.functional.binary_cross_entropy_with_logits(model(b), churned(b))
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            print(f"epoch {epoch}: loss {np.mean(losses):.4f}")
        task.to_artifact().save(workdir / "artifact")
        api.save_weights(workdir / "model.pt", model.state_dict(), task.fitted_state)

## Batch inference from the artifact

The artifact holds the contract and the fitted encoders; the weights file is yours. Here
both are loaded back as a fresh process would, and the test anchors are scored in
inference mode: no label rows are read and no batch carries targets.

In [ ]:
artifact = api.Artifact.load(workdir / "artifact")
scorer = ChurnModel(artifact.output_schema, NUMBERS, CATEGORIES)
scorer.load_state_dict(api.load_weights(workdir / "model.pt", artifact.fitted_state))
scorer.eval()

churn_risk: dict[int, float] = {}
with api.prepare(artifact.contract, dataset, engine) as plan:
    with plan.materialize(anchors, fitted=artifact.fitted_state, inference=True) as task:
        with torch.no_grad():
            for b in task.batches(split="test"):
                assert not b.targets
                for sample_id, risk, pad in zip(
                    b.metadata["sample_ids"],
                    torch.sigmoid(scorer(b)).tolist(),
                    b.metadata["is_padding"].tolist(),
                    strict=True,
                ):
                    if not pad:
                        churn_risk[int(sample_id)] = risk
print(f"scored {len(churn_risk)} test anchors")

## The leakage check

The generator records, for each anchor, the true probability of a purchase in the next
seven days given everything before the cutoff, and the AUROC that probability reaches. We
call that AUROC the **ceiling**. A model that reads only past rows can't beat it beyond
sampling noise. These true probabilities come from the data generator; a real dataset
wouldn't have them.

In [ ]:
oracle = pq.read_table(data.labels).to_pylist()
ids = [row["sample_id"] for row in oracle if row["sample_id"] in churn_risk]
did_churn = np.array([not row["label"] for row in oracle if row["sample_id"] in churn_risk])
area = datasets.auroc(did_churn, np.array([churn_risk[i] for i in ids]))
ceiling = data.metadata.ceiling_auroc["test"]
noise = data.metadata.ceiling_auroc_se["test"]
print(f"model test AUROC {area:.3f}; ceiling {ceiling:.3f} ± {noise:.3f}")

## Serving from caller-supplied rows

Online, the caller fetches rows and the library assembles the batch. `rows_spec` says
which tables, columns and time ranges one request needs; anything else is refused. The
lookup below is the caller's code: one customer's row and their transactions up to the
cutoff. The request served is the first test anchor whose customer has bought before.

In [ ]:
server = api.ServingAssembler(artifact)
print(server.rows_spec)

customers = pq.read_table(data.table("customers"))
transactions = pq.read_table(data.table("transactions"))


def needed(name: str) -> list[str]:
    return [c for c, column in server.rows_spec.tables[name].columns.items() if column.required]


def fetch(customer_id: int, cutoff: datetime) -> dict[str, pa.Table]:
    return {
        "customers": customers.filter(pc.equal(customers["customer_id"], customer_id)).select(
            needed("customers")
        ),
        "transactions": transactions.filter(
            pc.and_(
                pc.equal(transactions["customer_id"], customer_id),
                pc.less_equal(transactions["t_dat"], cutoff),
            )
        ).select(needed("transactions")),
    }


for request_row in pq.read_table(data.anchors).to_pylist():
    rows = fetch(request_row["customer_id"], request_row["cutoff"])
    if request_row["sample_id"] in churn_risk and rows["transactions"].num_rows > 0:
        break
request = api.SampleRequest(
    request_row["sample_id"], request_row["customer_id"], request_row["cutoff"]
)
(served,) = server.assemble(rows, [request])
with torch.no_grad():
    served_risk = torch.sigmoid(scorer(served)).item()
batch_risk = churn_risk[request.sample_id]
print(f"served {served_risk:.4f}; batch inference {batch_risk:.4f}")

## The files next to this notebook

The data, the DiskGraph index, the artifact and the weights stay in the temporary
directory. Three small files are written next to this notebook:

- `contract.json`: the contract, byte for byte the file the artifact stores, with the
  column types read from the Parquet files.
- `output_schema.json`: what the model receives, per input and field: dtype, vocabulary
  or buckets, fitted statistics. Floats are kept to six significant digits so the file is
  the same on every platform.
- `serving_request.json`: the request served above and the rows the caller supplied
  for it, timestamps in ISO 8601.

In [ ]:
Path("contract.json").write_text(artifact.contract.to_json() + "\n")
write_json("output_schema.json", rounded(artifact.output_schema))
write_json(
    "serving_request.json",
    {
        "request": {
            "sample_id": request.sample_id,
            "entity_id": request.entity_id,
            "as_of_time": request.as_of_time,
        },
        "rows": {name: table.to_pylist() for name, table in rows.items()},
    },
)
written = ("contract.json", "output_schema.json", "serving_request.json")
print({name: len(Path(name).read_bytes()) for name in written})

## Check

The model learned something (well above chance), stays below the ceiling (no leakage),
and serving reproduces batch inference.

In [ ]:
assert area > 0.75, area
assert area <= ceiling + 3 * noise, (area, ceiling)
assert abs(served_risk - batch_risk) < 1e-5
assert Path("contract.json").read_text() == (workdir / "artifact" / "contract.json").read_text()
print("checks passed")